In [ ]:
from pathlib import Path
import numpy as np
import pandas
import rasterio
import geopandas as gpd
import pandas as pd
import scipy
from rasterio.warp import calculate_default_transform, reproject, Resampling
import Robyn_river_floods
import re

In [ ]:
output_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info")
output_path.mkdir(parents=True, exist_ok=True)


In [ ]:
# import the catchment forest area statistics cvs
upstream_basin_cover = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info/basin_aff_ffe_areas.csv")


upstream_basin_cover = pd.read_csv(upstream_basin_cover)
display(upstream_basin_cover.head())          # or df.head()

In [ ]:
# Load interpolated peak flow reduction data
peak_flow_catchment_coverage_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data/interpolated_peak_flow_catchment_coverage.csv")
peak_flow_catchment_coverage = pandas.read_csv(peak_flow_catchment_coverage_path)

In [ ]:
peak_flow_catchment_coverage

#### Interpolate to a full table of catchment forest percentages and peak flow reductions for RP5 and 100

In [ ]:
# Create the initial dataframe with the original column name
peak_flow_catchment_coverage = pandas.DataFrame({
    'catchment_forest_percentage': [0, 6, 14, 21, 62, 100],
    'rp5.0': [0, 3, 13, 18, 48, 48],
    'rp100.0': [0, 1, 8, 11, 32, 32],
})
# peak_flow_catchment_coverage.rename(columns={'catchment_forest_percentage': 'Catchment forest coverage (%)'}, inplace=True)
display(peak_flow_catchment_coverage)

# Create a full range of percentages from 0 to 100
full_percentages = pandas.DataFrame({'catchment_forest_percentage': np.arange(0, 101)})

# Merge the full range with the existing data
interpolated_data = pandas.merge(full_percentages, peak_flow_catchment_coverage, on='catchment_forest_percentage', how='left')

# Perform linear interpolation to fill missing values
interpolated_data['rp5.0'] = interpolated_data['rp5.0'].interpolate(method='linear')
interpolated_data['rp100.0'] = interpolated_data['rp100.0'].interpolate(method='linear')

# interpolated_data.rename(columns={'catchment_forest_percentage': 'Catchment forest coverage (%)'}, inplace=True)
interpolated_data = interpolated_data.round(2)
interpolated_data.to_csv("interpolated_peak_flow_catchment_coverage.csv", index=False)
display(interpolated_data)

#### % Change in peak flow impact on return period

In [ ]:
def get_rp_cols(df):
    rp_cols = [col for col in df.columns if "rp" in col]
    rps = [float(col.replace("rp", "")) for col in rp_cols]
    return rp_cols, rps

def peak_flow_reduction(forest_perc, lookup):
    data = lookup.catchment_forest_percentage
    rp_cols, rps = get_rp_cols(lookup)
    interpolator = scipy.interpolate.RegularGridInterpolator(
        (rps, data),
        lookup[rp_cols].values.T,
        method='linear'
    )
    vals = interpolator(([rps], [forest_perc]))[0]
    return pandas.DataFrame({
        'catchment_forest_percentage': [forest_perc],
        'rp5.0': vals[0],
        'rp100.0': vals[1],
    })

peak_flow_reduction(7, peak_flow_catchment_coverage)

In [ ]:
# Top row outlines change in peak flow. Subsequent rows define the return period and what it becomes.
### Assume for return period of 2 or below, flood depth is 0, i.e. the infrastructure asset experiences no damage. 

""
"Need to add in 30% values - is there a way to interpolate this?"
""

CCRA_flow_reductions = pandas.DataFrame({
    'reduction_percent': [5, 10, 20, 40],
    #'rp2.0': [2.4, 3.2, 6.8, 196],
    #'rp2.3': [2.8, 3.7, 7.9, 169],
    'rp5.0': [6.5, 8.9, 19, 235],
    'rp10.0': [13,19,43,473],
    #'rp25.0': [35, 51, 123, 1330],
    'rp50.0': [72,107,268,2967],
    'rp100.0': [147,224,582, 6648],
    #'rp500.0': [772,1229,3441,43094],
    #'rp1000.0': [1571,2544,7345,95943],
})

# proportion of baseline flow
CCRA_flow_reductions['flow'] = (1 - CCRA_flow_reductions.reduction_percent / 100)

# The below code is used to interpolate based on the current return periods (2,12,50,100,500,1000) to define a new return period (20)
## interpolate RP20 values
known_rp_cols = [rp_col for rp_col in CCRA_flow_reductions.columns if "rp" in rp_col]
known_rps = [float(rp.replace("rp","")) for rp in known_rp_cols]
flows = CCRA_flow_reductions['flow'].values

interpolator = scipy.interpolate.RegularGridInterpolator(
    (known_rps, flows),
    CCRA_flow_reductions[known_rp_cols].values.T,
    method='cubic'
)

rps_new = [[20.0]]
CCRA_flow_reductions['rp20.0'] = interpolator((rps_new, [flows]))[0]

out_rps = sorted([20.0] + known_rps)
out_rp_cols = [f"rp{rp}" for rp in out_rps]

CCRA_flow_reductions_interpolated = CCRA_flow_reductions[['flow'] + out_rp_cols].copy()

flow07_row_values = interpolator(([out_rps], [[0.7]]))[0]
flow07_row = pandas.DataFrame(data=[[0.7] + list(flow07_row_values)], columns=['flow']+out_rp_cols)

CCRA_flow_reductions_interpolated = (
    pandas.concat([CCRA_flow_reductions_interpolated, flow07_row])
    .sort_values("flow", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
# for change in land cover what is the change in RP - should be bigger or the same if no forest is added

In [ ]:
# Define the known return periods (the ones you have columns for)
rp_known = [20, 50, 100, 200, 500, 1500]

# Define the full range to interpolate over (here every integer from 20 to 1500)
rp_full = np.arange(20, 1501)


In [ ]:
def interpolate_flow_reductions(current_cover_perc, future_cover_perc, flow_reductions_df, return_period):
    """
    Interpolates peak flow reductions based on current and future forest cover percentages.
    
    Parameters:
        current_cover_perc (float): The current forest cover percentage.
        future_cover_perc (float): The future forest cover percentage (e.g., after reforestation).
        flow_reductions_df (pd.DataFrame): DataFrame containing forest cover percentages and flow reductions.
        return_period (str): The column name in the DataFrame for the return period (e.g., 'rp5.0').
    
    Returns:
        tuple: (current_flow_reduction, future_flow_reduction) interpolated for the given return period.
    """
    # Ensure percentages are within bounds (of 0-100)
    current_cover_perc = max(0, min(100, current_cover_perc))
    future_cover_perc = max(0, min(100, future_cover_perc))
    
    # Interpolate flow reductions for the current and future percentages
    current_flow_reduction = np.interp(
        current_cover_perc,
        flow_reductions_df['catchment_forest_percentage'],
        flow_reductions_df[return_period]
    )
    
    future_flow_reduction = np.interp(
        future_cover_perc,
        flow_reductions_df['catchment_forest_percentage'],
        flow_reductions_df[return_period]
    )
    
    return current_flow_reduction, future_flow_reduction

upstream_basin_cover = upstream_basin_cover.copy()


# Define return periods to analyze
return_periods = ['rp5.0', 'rp100.0']

# Loop through each return period and calculate reductions
for return_period in return_periods:
    # 1) Interpolate current & future flow reductions
    upstream_basin_cover[f'current_flow_reduction_{return_period}'], upstream_basin_cover[f'future_flow_reduction_{return_period}'] = zip(*upstream_basin_cover.apply(
        lambda row: interpolate_flow_reductions(
            current_cover_perc=row['existing_forest_pct'],
            future_cover_perc=row['total_future_forest_pct'],
            flow_reductions_df=peak_flow_catchment_coverage,
            return_period=return_period
        ),
        axis=1
    ))
    
    # 2) Compute the ratio-based columns
    upstream_basin_cover[f'future_ratio_{return_period}'] = 100 - upstream_basin_cover[f'future_flow_reduction_{return_period}']
    upstream_basin_cover[f'current_ratio_{return_period}'] = 100 - upstream_basin_cover[f'current_flow_reduction_{return_period}']
    
    upstream_basin_cover[f'change_in_ratio_{return_period}'] = (
        upstream_basin_cover[f'current_ratio_{return_period}'] 
        - upstream_basin_cover[f'future_ratio_{return_period}']
    )
    
    upstream_basin_cover[f'future_reduction_proportion_{return_period}'] = (
        upstream_basin_cover[f'change_in_ratio_{return_period}'] 
        / upstream_basin_cover[f'current_ratio_{return_period}']
    ) * 100

# Now do rounding
columns_to_round = [col for col in upstream_basin_cover.columns if 'flow_reduction' in col or 'reduction_difference' in col]
columns_to_round += [col for col in upstream_basin_cover.columns if 'ratio' in col or 'future_reduction_proportion' in col]
upstream_basin_cover[columns_to_round] = upstream_basin_cover[columns_to_round].round(2)

# Debug
display(
    upstream_basin_cover[
        [
            'basin_file', 
            'existing_forest_pct', 
            'total_future_forest_pct'
        ] 
        + columns_to_round
    ].head()
)

# Export updated data to CSV for each return period
for return_period in return_periods:
# write ONE combined CSV with all columns
    combined_out = output_path / "peak_flow_reductions_by_basin.csv"
    upstream_basin_cover.to_csv(combined_out, index=False)
    print(f"Wrote {combined_out}")

In [ ]:
display(CCRA_flow_reductions_interpolated)

In [ ]:
# example data frame of pre-calculated total damages in each upstream basin for a set of return periods
damages = pandas.DataFrame(
    data={
        "basin_file": [123, 124, 125],
        "rp0.001": [0,0,0],
        "rp2.0": [0,0,0],
        "rp20.0": [10,20,30]
    }
).set_index("basin_file")

# Define output return periods
rps_to_calculate = [2.0, 5.0, 10.0]

# Call the calculate_rp_maps function
interpolated_damages = Robyn_river_floods.interpolate_rp_damages(rps_to_calculate, damages)
interpolated_damages

In [ ]:
Robyn_river_floods.peak_flow_reduction(forest_percentage_change=[0, 10, 100])

In [ ]:
rp_cols = ["rp5.0","rp10.0","rp20.0","rp50.0","rp100.0"]
catchment_peak_flow_reduction = Robyn_river_floods.peak_flow_reduction(upstream_basin_cover.afforestable_pct) \
    [rp_cols] \
    .rename(columns={rp_col: f"pfr_{rp_col}" for rp_col in rp_cols})

catchments_with_rp_change = upstream_basin_cover.join(catchment_peak_flow_reduction)
catchments_with_rp_change

for rp_col in rp_cols:
    rp_change = Robyn_river_floods.rp_change_given_flow_reduction(reduction_percent=catchments_with_rp_change[f"pfr_{rp_col}"], interp_rp=float(rp_col.replace("rp", ""))) \
        [[rp_col]] \
        .rename(columns={rp_col: f"future_{rp_col}"})
    catchments_with_rp_change = catchments_with_rp_change.join(rp_change)

catchments_with_rp_change

In [ ]:
# read split damages
# read flooded points snapped to river network  from merged_fid (done)
# for each flooded point
  # find the set of damaged (split) assets at that point (cell_index_2_x/y) or flood_i/j - few rows dataframe subset of split damages
  # find the catchment row at the related river network point (dem_i/j) - one row dataframe subset of catchments_with_rp_change
  # call adapted version of calculate_future_sector_damages  (without the hybas or variant for loops)

In [ ]:
damage_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data/direct_damages_fred/damages")


In [ ]:
# Example damage file
def read_damage_file(fname):
    example_damage = pd.read_parquet(fname) 
    example_damage.head()
    to_drop_col_names = [
        col for col in example_damage.columns 
        if col.startswith("coastal") 
        or col.startswith("surface") 
        or (col.startswith("fluvial") and "baseline" not in col)
    ]
    to_drop_cell_index_cols = [
        col for col in example_damage.columns 
        if col.startswith("cell_index") 
        and "2" not in col
    ]
    example_damage = example_damage.drop(columns=to_drop_col_names+to_drop_cell_index_cols)
    
    return example_damage

example_damage = read_damage_file("damage_dir/rail_edges_direct_damages_parameter_set_2.parquet").query('fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None > 0')
example_damage.columns, example_damage.shape

example_damage.head()

In [ ]:
# How to read every damage file in a loop, and pick out
# its asset class from the filename -- when you need to...

damage_paths = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data/direct_damages_fred/damages").glob("*.parquet")

# damage_paths = Path("../../Processed_data/direct_damages_fred/damages").glob("*.parquet")
# damage_paths = Path("damages").glob("*.parquet")

data = {}
for path in damage_paths:
    print(path)
    match = re.match(
        r"([A-Za-z0-9_.]+)_direct_damages_parameter_set_(\d+)\.parquet",
        Path(path).name
    )
    asset_class, ensemble_member = match.groups()
    data[(asset_class, ensemble_member)] = read_damage_file(path)

damage_paths

In [ ]:
import os
print(os.getcwd())

In [ ]:
flooded_points_snapped = pd.read_parquet("merged_fid_subset.parquet")

In [ ]:
flooded_points_snapped

In [ ]:
def calculate_future_sector_damages(sector_damages, catchments_with_rp_change):
    # variant_dfs = []
    # for variant in ['mean', 'amax', 'amin']:
    #     # select and rename your baseline damages to rpX.0, rpY.0, etc:
    #     selected_damages = Robyn_river_floods.select_damages(
    #         sector_damages, variant=variant
    #     ).reset_index()

    #     dfs = []
    #     for hybas_id, hybas in catchments_with_rp_change.iterrows():
    #         # pick out just this catchment’s baseline-damage row,
    #         # set HYBAS_ID into the index so it doesn’t get treated as an rp column
    #         hybas_damages = (
    #             selected_damages
    #             .loc[selected_damages.HYBAS_ID == hybas_id]
    #             .set_index("HYBAS_ID")
    #         )

# baseline RPs to estimate
interpolated_baseline_damages = Robyn_river_floods.interpolate_rp_damages(  # robyn import
    [5.0, 10.0],
    hybas_damages
)

baseline_damages = hybas_damages.join(interpolated_baseline_damages)

future_rps = [float(colname.replace("future_rp", "")) for colname in catchments_with_rp_change.columns if "future_rp" in colname]

# get the *future* RP values corresponding to each
current_to_future_rps = {
    rp: hybas[f"future_rp{float(rp)}"]
    for rp in future_rps
}
# display(current_to_future_rps)

# display(baseline_damages)

future_rp_columns = [f"rp{rp}" for rp in future_rps]
future_damages = baseline_damages[future_rp_columns].rename(columns={
    f"rp{current}": f"rp{future}"
    for current, future in current_to_future_rps.items()
})

extreme_damages = baseline_damages[["rp0.0001", "rp1000000000.0"]].copy()
future_damages = future_damages.join(extreme_damages)
future_damages_columns = sorted(
    future_damages.columns, 
    key=lambda c: float(c.replace("rp", ""))
)
future_damages = future_damages[future_damages_columns].copy()
# at this point future damages contains columns for each of the adjusted future return periods
# corresponding to the initial baseline set (e.g. rp6.4, rp12.0, rp24.1 corresponding to what was
# rp5.0, rp10.0, rp20.0) - these could be used to calculate EAD directly, without the next bit
# of interpolation.

# could call Robyn_river_floods.calculate_ead here using future_damages

# display(future_damages)
# calculate EAD for this catchment - append to interpolated damages further down
# 5) calculate EAD directly on these shifted‐RP columns
# ead_values = Robyn_river_floods.calculate_ead(future_damages)
# ead_colname = f"future__fluvial__ead__{variant}" 
baseline_ead_values = Robyn_river_floods.calculate_ead(hybas_damages)
baseline_ead_colname = f"baseline__fluvial__ead__{variant}" 


# perform the interpolation 
interpolated_hybas_damages = Robyn_river_floods.interpolate_rp_damages(
    future_rps,
    future_damages
)
# display(interpolated_hybas_damages)
ead_values = Robyn_river_floods.calculate_ead(interpolated_hybas_damages)
ead_colname = f"future__fluvial__ead__{variant}" 

# now rename its *output* columns (one per rp_to_calculate)
to_rename = {
    f"rp{rp}": f"future__fluvial__rp_{int(rp)}__future__fluvial__rp_{variant}"
    for rp in future_rps
}
interpolated_hybas_damages.rename(columns=to_rename, inplace=True)

# pick out the interpolated future rp damages
interpolated_hybas_damages = interpolated_hybas_damages[to_rename.values()]
# add future ead as calculated above
interpolated_hybas_damages[ead_colname] = ead_values

# rename from rpXX columns to baseline...
to_rename = {
    f"rp{rp}": f"baseline__fluvial__rp_{int(rp)}__baseline__fluvial__rp_{variant}"
    for rp in future_rps
}
baseline_damages_renamed = baseline_damages.rename(columns=to_rename)[to_rename.values()]
baseline_damages_renamed[baseline_ead_colname] = baseline_ead_values
    
# join the interpolated baseline rp damages
interpolated_hybas_damages = pandas.concat([
    interpolated_hybas_damages,
    baseline_damages_renamed
], axis=1)

# re‑attach the index as a column
interpolated_hybas_damages["HYBAS_ID"] = hybas_id
dfs.append(interpolated_hybas_damages)

variant_damages = pandas.concat(dfs).set_index("HYBAS_ID")
variant_dfs.append(variant_damages)

# concatenate all three variants (mean, amax, amin) side‑by‑side
return pandas.concat(variant_dfs, axis=1)
# future_damages#.loc[7120065500, "future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean"]